# Implementation demonstration

We ask our two fine-tuned models the same questions and compare their answers. Both have about 47M
parameters, were trained from scratch on FineWeb-Edu and then instruction-tuned on Alpaca-cleaned
(experiment E7). Answers use greedy decoding with at most 60 new tokens, the same as in our evaluation.
Nothing is filtered or picked: whatever the models write is printed.

**Kaggle:** Add Input → Notebook Output → `afml-sft` (the fine-tuned models) and `afml-data` (the
tokenizer). A CPU is enough (Accelerator: None). Internet must be on, to clone the code.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = 'https://github.com/Gagan-1718/afml-transformer-vs-state-space-models'
INPUT = Path(os.environ.get('AFML_DEMO_INPUT', '/kaggle/input'))  # where the model files are

# The code: this repository if the notebook runs inside it, otherwise a fresh clone.
repo = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'afml' / 'sft.py').exists()), None)
if repo is None:
    repo = Path('afml-code')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo)], check=True)
sys.path.insert(0, str(repo.resolve()))

import torch
from tokenizers import Tokenizer
from afml.eval.common import load_model
from afml.models import count_params
from afml.sft import PROMPT, PROMPT_WITH_INPUT, greedy


def find(pattern):
    hits = sorted(glob.glob(str(INPUT / '**' / pattern), recursive=True))
    if not hits:
        raise FileNotFoundError(f'{pattern} not found under {INPUT}. Add the afml-sft and afml-data '
                                'outputs with "Add Input".')
    return Path(hits[0])


chosen = json.loads((repo / 'results' / 'sft' / 'best.json').read_text())  # chosen on validation
tok = Tokenizer.from_file(str(find('tokenizer.json')))
EOS = tok.token_to_id('[EOS]')
models = {}
for label, arch in (('Transformer', 'transformer'), ('Mamba', 'mamba')):
    run = find(f'sft/{chosen[arch]}/best.pt').parent
    models[label] = load_model(run, device='cpu', scan='chunked')  # our PyTorch scan; no GPU needed
    print(f'{label:<12} {chosen[arch]:<28} {count_params(models[label]) / 1e6:.1f}M parameters')

In [ ]:
def ask(question, text=''):
    """Ask both models the same question. text: optional input, e.g. a review to classify."""
    prompt = (PROMPT_WITH_INPUT if text else PROMPT).format(instruction=question, input=text)
    ids = tok.encode(prompt).ids
    print('Q:', question)
    if text:
        print('   Input:', text)
    for label, model in models.items():
        start = time.time()
        with torch.no_grad():
            out, stopped = greedy(model, ids, 60, EOS, 'cpu')
        answer = ' '.join(tok.decode(out).split()) + ('' if stopped else ' ...')  # on one line
        print(f'   {label:<12} ({time.time() - start:.1f} s): {answer}')
    print()

In [ ]:
ask('What is the capital of India?')
ask('Name two benefits of regular exercise.')
ask('Classify the sentiment of this review as positive or negative.',
    'I loved this movie, the story was wonderful.')
ask('What is 5 plus 3?')

## Your own question
Run the cell and type a question (or edit `QUESTION` and run it again).

In [ ]:
QUESTION = 'Write one sentence about the moon.'
try:
    QUESTION = input('Your question: ').strip() or QUESTION
except Exception:  # no keyboard in a background run ("Save & Run All"): use QUESTION as written
    pass
ask(QUESTION)